In [1]:
"""
=============================================================================
QUANTITATIVE EVALUATION - Mental Health NLP Models
=============================================================================
Metrics: Recall (per-class), Macro-F1, PR-AUC (Precision-Recall AUC)
Models:  SVM, Bi-LSTM, BERT
Dataset: 7-class imbalanced mental health dataset (~103K samples)

Run on Kaggle:
  - Input dataset:  /kaggle/input/datasets/dx9029/mental-health/
  - Input models:   /kaggle/input/<your-models-dataset>/
  - Output:         /kaggle/working/
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os
import gc
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")           # headless on Kaggle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.svm import LinearSVC
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")


In [2]:
! ls -l /kaggle/input/datasets

total 4
drwxr-xr-x 4 root root 4096 May 21 16:36 dx9029


In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS  ← adjust dataset slugs as needed
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("/kaggle/input/datasets/dx9029/mental-health")
MODEL_DIR  = Path("/kaggle/input/datasets/dx9029/mental-health-models")   # your saved models dataset
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

BERT_MODEL_NAME = "mental-bert-base-uncased"   # local path inside MODEL_DIR or HF hub name
BERT_MAX_LEN    = 128
BERT_BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ["anxiety", "bipolar", "depression", "normal",
               "personality disorder", "stress", "suicidal"]
NUM_CLASSES = len(CLASS_NAMES)

# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD PREPROCESSED DATA
# ─────────────────────────────────────────────────────────────────────────────
def load_data():
    print("=" * 70)
    print("LOADING PREPROCESSED FEATURES")
    print("=" * 70)

    with open(DATA_DIR / "X_test_scaled.pkl", "rb") as f:
        X_test = pickle.load(f)
    with open(DATA_DIR / "y_test.pkl", "rb") as f:
        y_test = pickle.load(f)
    with open(DATA_DIR / "label_mapping.pkl", "rb") as f:
        label_mapping = pickle.load(f)
    # Raw text needed for BERT
    test_df = pd.read_csv(DATA_DIR / "test_processed.csv")

    print(f"✓ X_test shape : {X_test.shape}")
    print(f"✓ y_test shape : {y_test.shape}")
    print(f"✓ Classes      : {label_mapping}")
    print(f"✓ Text samples : {len(test_df)}")
    print()
    return X_test, y_test, label_mapping, test_df

In [2]:

# ─────────────────────────────────────────────────────────────────────────────
# 4. BI-LSTM ARCHITECTURE (must match training definition)
# ─────────────────────────────────────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=3,
                 num_classes=7, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ─────────────────────────────────────────────────────────────────────────────
# 5. PREDICTION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def predict_svm(X_test):
    """Load saved SVM and return hard predictions."""
    svm_path = MODEL_DIR / "svm_model_full.pkl"
    print(f"  Loading SVM from {svm_path}")
    with open(svm_path, "rb") as f:
        model = pickle.load(f)
    preds = model.predict(X_test)
    # LinearSVC has decision_function but no predict_proba;
    # use calibrated scores for PR-AUC
    scores = model.decision_function(X_test)   # shape (n, 7)
    del model; gc.collect()
    return preds, scores


def predict_bilstm(X_test):
    """Load saved Bi-LSTM and return hard predictions + softmax probabilities."""
    lstm_path = MODEL_DIR / "lstm_model_full.pt"
    print(f"  Loading Bi-LSTM from {lstm_path}")

    X_tensor = torch.FloatTensor(
        X_test.toarray() if hasattr(X_test, "toarray") else X_test
    ).unsqueeze(1)   # (N, 1, 300)

    model = BiLSTMClassifier(input_size=X_tensor.shape[2]).to(DEVICE)
    model.load_state_dict(torch.load(lstm_path, map_location=DEVICE))
    model.eval()

    loader = DataLoader(TensorDataset(X_tensor), batch_size=256, shuffle=False)
    all_probs, all_preds = [], []

    with torch.no_grad():
        for (batch,) in loader:
            logits = model(batch.to(DEVICE))
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(1).cpu().numpy()
            all_probs.append(probs)
            all_preds.append(preds)

    del model; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(all_preds), np.concatenate(all_probs)


def predict_bert(test_df, text_col="statement"):
    """Load saved BERT model and return hard predictions + softmax probs."""
    MODEL_DIR = Path("/kaggle/input/datasets/dx9029/model_bert")
    bert_path = MODEL_DIR / BERT_MODEL_NAME
    if not bert_path.exists():
        if (MODEL_DIR / "bert_model_full").exists():
            bert_path = MODEL_DIR / "bert_model_full"
        elif (MODEL_DIR / "config.json").exists():
            bert_path = MODEL_DIR
        else:
            config_files = list(MODEL_DIR.glob("**/config.json"))
            if config_files:
                bert_path = config_files[0].parent
            else:
                bert_path = Path("mental/mental-bert-base-uncased")

    print(f"  Loading BERT from {bert_path}")

    try:
        model = AutoModelForSequenceClassification.from_pretrained(str(bert_path), num_labels=NUM_CLASSES).to(DEVICE)
        tokenizer = AutoTokenizer.from_pretrained(str(bert_path))
    except Exception as e:
        print(f"    ⚠️ Local load failed ({e}), falling back to HuggingFace Hub...")
        hub_id = "mental/mental-bert-base-uncased"
        model = AutoModelForSequenceClassification.from_pretrained(hub_id, num_labels=NUM_CLASSES).to(DEVICE)
        tokenizer = AutoTokenizer.from_pretrained(hub_id)
    model.eval()

    texts = test_df[text_col].fillna("").tolist()
    all_probs, all_preds = [], []

    for i in range(0, len(texts), BERT_BATCH_SIZE):
        batch_texts = texts[i : i + BERT_BATCH_SIZE]
        enc = tokenizer(
            batch_texts,
            max_length=BERT_MAX_LEN,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(preds)
        if (i // BERT_BATCH_SIZE) % 20 == 0:
            print(f"    BERT progress: {min(i+BERT_BATCH_SIZE, len(texts))}/{len(texts)}")

    del model; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(all_preds), np.concatenate(all_probs)



SyntaxError: unterminated string literal (detected at line 74) (1030966926.py, line 74)

In [5]:

# ─────────────────────────────────────────────────────────────────────────────
# 6. METRICS COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────
def compute_pr_auc(y_true, y_scores):
    """
    Compute per-class and macro PR-AUC for multi-class.
    y_scores: (N, C) probability / decision-function scores.
    """
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

    # Normalize decision scores to [0,1] range if needed (e.g. LinearSVC)
    if y_scores.min() < 0 or y_scores.max() > 1:
        from sklearn.preprocessing import MinMaxScaler
        y_scores = MinMaxScaler().fit_transform(y_scores)

    pr_auc_per_class = {}
    for i, name in enumerate(CLASS_NAMES):
        ap = average_precision_score(y_bin[:, i], y_scores[:, i])
        pr_auc_per_class[name] = ap

    macro_pr_auc = np.mean(list(pr_auc_per_class.values()))
    return pr_auc_per_class, macro_pr_auc


def evaluate_model(model_name, y_true, y_pred, y_scores):
    """Compute and return a dict of all required metrics."""
    print(f"\n{'='*70}")
    print(f"EVALUATION: {model_name}")
    print(f"{'='*70}")

    acc       = accuracy_score(y_true, y_pred)
    macro_f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    w_f1      = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
    prec_per_class   = precision_score(y_true, y_pred, average=None, zero_division=0)
    pr_auc_per_class, macro_pr_auc = compute_pr_auc(y_true, y_scores)

    results = {
        "model"           : model_name,
        "accuracy"        : acc,
        "macro_f1"        : macro_f1,
        "weighted_f1"     : w_f1,
        "macro_pr_auc"    : macro_pr_auc,
    }

    for i, name in enumerate(CLASS_NAMES):
        results[f"recall_{name}"]  = recall_per_class[i]
        results[f"prec_{name}"]    = prec_per_class[i]
        results[f"pr_auc_{name}"]  = pr_auc_per_class[name]

    # Console summary
    print(f"  Accuracy       : {acc:.4f}")
    print(f"  Macro-F1       : {macro_f1:.4f}")
    print(f"  Weighted-F1    : {w_f1:.4f}")
    print(f"  Macro PR-AUC   : {macro_pr_auc:.4f}")
    print(f"\n  Per-class Recall:")
    for i, name in enumerate(CLASS_NAMES):
        flag = " ⚠️" if name in ("suicidal", "depression") else ""
        print(f"    {name:25s}: Recall={recall_per_class[i]:.4f}  PR-AUC={pr_auc_per_class[name]:.4f}{flag}")

    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    return results


# ─────────────────────────────────────────────────────────────────────────────
# 7. VISUALISATION
# ─────────────────────────────────────────────────────────────────────────────
PALETTE = {"SVM": "#4C72B0", "Bi-LSTM": "#55A868", "BERT": "#C44E52"}

def plot_comparison_bar(df_summary, output_dir):
    """Bar chart: Accuracy, Macro-F1, Macro-PR-AUC side-by-side per model."""
    metrics = ["accuracy", "macro_f1", "macro_pr_auc"]
    labels  = ["Accuracy", "Macro F1-Score", "Macro PR-AUC"]
    x = np.arange(len(metrics))
    width = 0.22
    colors = [PALETTE[m] for m in df_summary["model"]]

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (_, row) in enumerate(df_summary.iterrows()):
        vals = [row[m] for m in metrics]
        bars = ax.bar(x + i * width, vals, width, label=row["model"],
                      color=list(PALETTE.values())[i], alpha=0.9, edgecolor="white")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Model Comparison — Key Metrics", fontsize=14, fontweight="bold", pad=15)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    path = output_dir / "01_metric_comparison.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Saved: {path}")


def plot_recall_heatmap(df_summary, output_dir):
    """Heatmap of per-class Recall for all 3 models."""
    recall_cols = [f"recall_{c}" for c in CLASS_NAMES]
    data = df_summary.set_index("model")[recall_cols]
    data.columns = CLASS_NAMES

    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(
        data.astype(float), annot=True, fmt=".3f", cmap="RdYlGn",
        vmin=0, vmax=1, linewidths=0.5, ax=ax,
        annot_kws={"size": 10, "weight": "bold"},
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title("Per-Class Recall — SVM vs Bi-LSTM vs BERT", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Mental Health Category", fontsize=11)
    ax.set_ylabel("Model", fontsize=11)
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

    # Highlight critical classes
    for j, name in enumerate(CLASS_NAMES):
        if name in ("suicidal", "depression"):
            ax.add_patch(plt.Rectangle((j, 0), 1, len(df_summary),
                                       fill=False, edgecolor="gold", lw=2.5))

    plt.tight_layout()
    path = output_dir / "02_recall_heatmap.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Saved: {path}")


def plot_pr_auc_heatmap(df_summary, output_dir):
    """Heatmap of per-class PR-AUC for all 3 models."""
    cols = [f"pr_auc_{c}" for c in CLASS_NAMES]
    data = df_summary.set_index("model")[cols]
    data.columns = CLASS_NAMES

    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(
        data.astype(float), annot=True, fmt=".3f", cmap="Blues",
        vmin=0, vmax=1, linewidths=0.5, ax=ax,
        annot_kws={"size": 10},
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title("Per-Class PR-AUC — SVM vs Bi-LSTM vs BERT", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Mental Health Category", fontsize=11)
    ax.set_ylabel("Model", fontsize=11)
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    path = output_dir / "03_pr_auc_heatmap.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Saved: {path}")


def plot_pr_curves(all_scores, y_true, output_dir):
    """
    Precision-Recall curves for critical classes (Suicidal & Depression)
    overlaid for all 3 models.
    """
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    critical = {"suicidal": 6, "depression": 2}

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (cls_name, cls_idx) in zip(axes, critical.items()):
        for model_name, scores in all_scores.items():
            s = scores.copy()
            if s.min() < 0 or s.max() > 1:
                from sklearn.preprocessing import MinMaxScaler
                s = MinMaxScaler().fit_transform(s)
            prec, rec, _ = precision_recall_curve(y_bin[:, cls_idx], s[:, cls_idx])
            ap = average_precision_score(y_bin[:, cls_idx], s[:, cls_idx])
            ax.plot(rec, prec, label=f"{model_name} (AP={ap:.3f})",
                    color=PALETTE[model_name], linewidth=2)

        ax.set_xlabel("Recall", fontsize=11)
        ax.set_ylabel("Precision", fontsize=11)
        ax.set_title(f"PR Curve — {cls_name.capitalize()}", fontsize=12, fontweight="bold")
        ax.legend(fontsize=9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
        ax.grid(alpha=0.3, linestyle="--")
        ax.spines[["top", "right"]].set_visible(False)

    plt.suptitle("Precision-Recall Curves for Critical Classes", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = output_dir / "04_pr_curves_critical.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Saved: {path}")


def plot_confusion_matrices(all_preds, y_true, output_dir):
    """3-panel confusion matrices (normalised)."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    short_names = ["Anx", "Bip", "Dep", "Nor", "PD", "Str", "Sui"]

    for ax, (model_name, y_pred) in zip(axes, all_preds.items()):
        cm = confusion_matrix(y_true, y_pred, normalize="true")
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                    xticklabels=short_names, yticklabels=short_names,
                    ax=ax, cbar=False, annot_kws={"size": 8})
        ax.set_title(f"{model_name}\n(row-normalised)", fontsize=11, fontweight="bold")
        ax.set_xlabel("Predicted", fontsize=9)
        ax.set_ylabel("True", fontsize=9)

    plt.suptitle("Confusion Matrices — SVM vs Bi-LSTM vs BERT", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = output_dir / "05_confusion_matrices.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Saved: {path}")


def plot_critical_recall_bar(df_summary, output_dir):
    """
    Grouped bar: Suicidal + Depression Recall for the 3 models.
    Key clinical insight chart.
    """
    classes  = ["suicidal", "depression"]
    x = np.arange(len(classes))
    width = 0.25

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, (_, row) in enumerate(df_summary.iterrows()):
        vals = [row[f"recall_{c}"] for c in classes]
        bars = ax.bar(x + i * width, vals, width,
                      label=row["model"], color=list(PALETTE.values())[i],
                      alpha=0.9, edgecolor="white")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticks(x + width)
    ax.set_xticklabels(["Suicidal Recall", "Depression Recall"], fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Recall (Sensitivity)", fontsize=12)
    ax.set_title("Critical Class Recall — Suicidal & Depression\n"
                 "(Higher = Fewer dangerous false negatives)", fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.axhline(0.7, color="orange", linestyle="--", linewidth=1.2, label="70% threshold")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    path = output_dir / "06_critical_recall.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Saved: {path}")



# MODEL SAVED

In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# 8. SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────
def save_results(all_results, output_dir):
    df = pd.DataFrame(all_results)
    csv_path = output_dir / "quantitative_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n✓ Results table saved: {csv_path}")

    # Pretty summary table
    summary_cols = ["model", "accuracy", "macro_f1", "weighted_f1", "macro_pr_auc",
                    "recall_suicidal", "recall_depression",
                    "pr_auc_suicidal", "pr_auc_depression"]
    print("\n" + "="*90)
    print("FINAL COMPARISON TABLE")
    print("="*90)
    print(df[summary_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("="*90)
    return df



# RUN

In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# 9. MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def main():
    global y_test, svm_preds, lstm_preds, bert_preds, texts
    print("\n" + "🔬 "*20)
    print("MENTAL HEALTH NLP — QUANTITATIVE EVALUATION")
    print("🔬 "*20 + "\n")

    # ── Load data ────────────────────────────────────────────────────────────
    X_test, y_test, label_mapping, test_df = load_data()

    # ── SVM ──────────────────────────────────────────────────────────────────
    print("\n[1/3] SVM ...")
    svm_preds, svm_scores = predict_svm(X_test)

    # ── Bi-LSTM ──────────────────────────────────────────────────────────────
    print("\n[2/3] Bi-LSTM ...")
    lstm_preds, lstm_scores = predict_bilstm(X_test)

    # ── BERT ─────────────────────────────────────────────────────────────────
    print("\n[3/3] BERT ...")
    # Detect text column name flexibly
    text_col = "statement" if "statement" in test_df.columns else test_df.columns[0]
    bert_preds, bert_scores = predict_bert(test_df, text_col=text_col)
    
    # Extract texts for edge case selection
    texts = test_df[text_col].fillna("").tolist()

    # ── Align lengths (BERT uses raw text, others use X_test) ──────────────
    n = min(len(y_test), len(svm_preds), len(lstm_preds), len(bert_preds))
    y_test      = np.array(y_test)[:n]
    svm_preds   = svm_preds[:n];   svm_scores  = svm_scores[:n]
    lstm_preds  = lstm_preds[:n];  lstm_scores = lstm_scores[:n]
    bert_preds  = bert_preds[:n];  bert_scores = bert_scores[:n]
    texts       = texts[:n]

    # ── Evaluate ─────────────────────────────────────────────────────────────
    all_results = []
    all_results.append(evaluate_model("SVM",     y_test, svm_preds,  svm_scores))
    all_results.append(evaluate_model("Bi-LSTM", y_test, lstm_preds, lstm_scores))
    all_results.append(evaluate_model("BERT",    y_test, bert_preds, bert_scores))

    # ── Summary table ────────────────────────────────────────────────────────
    df_summary = save_results(all_results, OUTPUT_DIR)

    # ── Visualisations ───────────────────────────────────────────────────────
    print("\n📊 Generating visualisations ...")
    all_preds_dict  = {"SVM": svm_preds,  "Bi-LSTM": lstm_preds, "BERT": bert_preds}
    all_scores_dict = {"SVM": svm_scores, "Bi-LSTM": lstm_scores, "BERT": bert_scores}

    plot_comparison_bar(df_summary, OUTPUT_DIR)
    plot_recall_heatmap(df_summary, OUTPUT_DIR)
    plot_pr_auc_heatmap(df_summary, OUTPUT_DIR)
    plot_pr_curves(all_scores_dict, y_test, OUTPUT_DIR)
    plot_confusion_matrices(all_preds_dict, y_test, OUTPUT_DIR)
    plot_critical_recall_bar(df_summary, OUTPUT_DIR)

    print("\n✅ EVALUATION COMPLETE")
    print(f"   All outputs saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 
MENTAL HEALTH NLP — QUANTITATIVE EVALUATION
🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 🔬 

LOADING PREPROCESSED FEATURES
✓ X_test shape : (15103, 300)
✓ y_test shape : (15103,)
✓ Classes      : {'anxiety': np.int64(0), 'bipolar': np.int64(1), 'depression': np.int64(2), 'normal': np.int64(3), 'personality disorder': np.int64(4), 'stress': np.int64(5), 'suicidal': np.int64(6)}
✓ Text samples : 15103


[1/3] SVM ...
  Loading SVM from /kaggle/input/datasets/dx9029/mental-health-models/svm_model_full.pkl

[2/3] Bi-LSTM ...
  Loading Bi-LSTM from /kaggle/input/datasets/dx9029/mental-health-models/lstm_model_full.pt

[3/3] BERT ...
  Loading BERT from /kaggle/input/datasets/dx9029/mental-health-models/mental-bert-base-uncased


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/input/datasets/dx9029/mental-health-models/mental-bert-base-uncased'. Use `repo_type` argument if needed.

# LLM-AS-A-JUDGE: Extract 50 Edge Cases for Manual Testing
Since you want to test 50 samples manually with an LLM, this section will:
1. Select 50 edge-case samples (e.g., where SVM failed but BERT succeeded).
2. Generate a text file (`llm_judge_batch_prompt.txt`) containing the prompts for all 50 samples.
3. You can then download this text file, copy-paste it into ChatGPT or Claude.
4. Save the LLM's JSON response to a file and upload it back to Kaggle.

In [ ]:
import random
import json

N_SAMPLES = 200
RANDOM_SEED = 42
CRITICAL_CLASSES = {"suicidal", "depression", "personality disorder"}

def select_edge_cases(texts, y_true, svm_preds, lstm_preds, bert_preds, n=N_SAMPLES, seed=RANDOM_SEED):
    rng = random.Random(seed)
    np.random.seed(seed)
    
    df = pd.DataFrame({
        "text"       : texts,
        "true_label" : [CLASS_NAMES[i] for i in y_true],
        "true_idx"   : y_true,
        "svm_pred"   : [CLASS_NAMES[i] for i in svm_preds],
        "lstm_pred"  : [CLASS_NAMES[i] for i in lstm_preds],
        "bert_pred"  : [CLASS_NAMES[i] for i in bert_preds],
    })
    
    svm_wrong  = svm_preds  != y_true
    lstm_wrong = lstm_preds != y_true
    bert_wrong = bert_preds != y_true
    bert_right = bert_preds == y_true
    all_wrong  = svm_wrong & lstm_wrong & bert_wrong
    is_critical = df["true_label"].isin(CRITICAL_CLASSES)
    
    # Tier 1: SVM wrong, BERT right
    tier1_mask = svm_wrong & bert_right
    tier1_crit = df[tier1_mask & is_critical].index.tolist()
    tier1_rest = df[tier1_mask & ~is_critical].index.tolist()
    rng.shuffle(tier1_crit); rng.shuffle(tier1_rest)
    tier1_pool = tier1_crit + tier1_rest
    tier1_idx = tier1_pool[:int(n * 0.60)]
    
    # Tier 2: All 3 wrong
    tier2_pool = [i for i in df[all_wrong].index.tolist() if i not in tier1_idx]
    rng.shuffle(tier2_pool)
    tier2_idx = tier2_pool[:int(n * 0.25)]
    
    # Tier 3: Critical class, at least 1 wrong
    already_chosen = set(tier1_idx) | set(tier2_idx)
    one_wrong = svm_wrong | lstm_wrong | bert_wrong
    tier3_pool = [i for i in df[is_critical & one_wrong].index.tolist() if i not in already_chosen]
    rng.shuffle(tier3_pool)
    tier3_idx = tier3_pool[:n - len(tier1_idx) - len(tier2_idx)]
    
    all_idx = tier1_idx + tier2_idx + tier3_idx
    if len(all_idx) < n:
        remaining = [i for i in df[svm_wrong].index.tolist() if i not in set(all_idx)]
        rng.shuffle(remaining)
        all_idx += remaining[:n - len(all_idx)]
        
    selected = df.loc[all_idx].copy().reset_index(drop=True)
    
    tier_map = {}
    for idx in tier1_idx: tier_map[idx] = "tier1_svm_wrong_bert_right"
    for idx in tier2_idx: tier_map[idx] = "tier2_all_models_wrong"
    for idx in tier3_idx: tier_map[idx] = "tier3_critical_class"
    selected["selection_tier"] = [tier_map.get(i, "tier4_fallback") for i in all_idx]
    
    selected["svm_correct"]  = selected["svm_pred"]  == selected["true_label"]
    selected["lstm_correct"] = selected["lstm_pred"]  == selected["true_label"]
    selected["bert_correct"] = selected["bert_pred"]  == selected["true_label"]
    
    print(f"Selected {len(selected)} samples.")
    return selected


In [ ]:
SYSTEM_PROMPT = """You are an expert clinical psychologist and NLP evaluation specialist.
Your task is to assess mental health text classification predictions made by three AI models
(SVM, Bi-LSTM, BERT) and provide structured, evidence-based evaluations.

You will evaluate each sample on four dimensions:
1. **Clinical Accuracy** (0–10): Does the predicted label reflect the clinical/psychological meaning of the text?
2. **Linguistic Complexity** (Low/Medium/High): Does the text use metaphors, sarcasm, euphemisms, or indirect language?
3. **Missed Risk Signals** (0–10): How severe would the consequences be if a real system made this error?
4. **Model Reasoning Analysis**: Why do you think SVM failed but BERT succeeded (or all three failed)?

Always respond in the exact JSON schema provided."""

def build_prompt(row, sample_id):
    prompt = f"""
## Sample #{sample_id + 1}

**Text to evaluate:**
"{row['text']}"

**Ground Truth Label:** `{row['true_label']}`

**Model Predictions:**
| Model   | Prediction          | Correct? |
|---------|---------------------|----------|
| SVM     | `{row['svm_pred']}` | {'✅' if row['svm_correct'] else '❌'} |
| Bi-LSTM | `{row['lstm_pred']}` | {'✅' if row['lstm_correct'] else '❌'} |
| BERT    | `{row['bert_pred']}` | {'✅' if row['bert_correct'] else '❌'} |

**Selection Reason:** {row['selection_tier'].replace('_', ' ').title()}

Please evaluate this sample and respond ONLY with the following JSON schema:

```json
{{
  "sample_id": {sample_id + 1},
  "ground_truth": "{row['true_label']}",
  "clinical_accuracy": {{
    "svm_score": <int 0-10>,
    "lstm_score": <int 0-10>,
    "bert_score": <int 0-10>
  }},
  "linguistic_complexity": "<Low|Medium|High>",
  "linguistic_features": ["<feature1>", "<feature2>"],
  "missed_risk_severity": <int 0-10>,
  "model_reasoning": "<2-3 sentences explaining WHY models succeeded or failed>",
  "clinical_insight": "<1-2 sentences from a psychologist's perspective>",
  "verdict": "<which model is most reliable for this sample and why>"
}}
```
""".strip()
    return prompt

def build_batch_prompt(selected_df):
    header = f"""{SYSTEM_PROMPT}

---

You will now evaluate {len(selected_df)} mental health text classification samples.
The 7 possible labels are: {', '.join(f'`{c}`' for c in CLASS_NAMES)}.

For each sample, respond with a JSON object following the schema shown in the first sample.
Wrap ALL responses in a top-level JSON array like:
```json
[
  {{ ... sample 1 ... }},
  {{ ... sample 2 ... }},
  ...
]
```

---
"""
    samples = "\n\n---\n\n".join(build_prompt(row, i) for i, row in selected_df.iterrows())
    return header + samples


In [ ]:
# Ensure predictions are available before generating
print("[1/3] Selecting 50 edge-case samples ...")
selected_df = select_edge_cases(texts, y_test, svm_preds, lstm_preds, bert_preds)

print("\n[2/3] Building LLM prompts ...")
batch_prompt = build_batch_prompt(selected_df)

print("\n[3/3] Saving outputs ...")
csv_path = OUTPUT_DIR / "llm_judge_samples.csv"
selected_df.to_csv(csv_path, index=False)
print(f"  ✓ Samples CSV     : {csv_path}")

prompt_path = OUTPUT_DIR / "llm_judge_batch_prompt.txt"
prompt_path.write_text(batch_prompt, encoding="utf-8")
print(f"  ✓ Batch prompt    : {prompt_path}")

print("""
╔══════════════════════════════════════════════════════════╗
║  NEXT STEPS FOR MANUAL LLM TESTING                       ║
║                                                          ║
║  1. Download:  llm_judge_batch_prompt.txt                ║
║  2. Paste its contents into GPT-4o / Claude chat         ║
║  3. Save the JSON response to a file locally             ║
║  4. Upload that JSON file back to Kaggle as a dataset    ║
╚══════════════════════════════════════════════════════════╝
""")


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Đường dẫn tới file JSON bạn vừa upload 
# (LƯU Ý: Đổi 'mental-health-models' thành tên dataset bạn vừa tạo trên Kaggle nếu khác)
json_path = Path("/kaggle/input/datasets/dx9029/mental-health/llm_judge_evaluations.json")
output_dir = Path("/kaggle/working")

if json_path.exists():
    with open(json_path, "r", encoding="utf-8") as f:
        evals = json.load(f)
    
    flat = []
    for ev in evals:
        # Lấy điểm clinical_accuracy của từng model
        ca = ev.get("clinical_accuracy", {})
        flat.append({
            "sample_id"            : ev.get("sample_id"),
            "ground_truth"         : ev.get("ground_truth"),
            "linguistic_complexity": ev.get("linguistic_complexity"),
            "missed_risk_severity" : ev.get("missed_risk_severity"),
            "svm_clinical_score"   : ca.get("svm_score"),
            "lstm_clinical_score"  : ca.get("lstm_score"),
            "bert_clinical_score"  : ca.get("bert_score"),
            "linguistic_features"  : "; ".join(ev.get("linguistic_features", [])),
            "model_reasoning"      : ev.get("model_reasoning"),
            "clinical_insight"     : ev.get("clinical_insight"),
            "verdict"              : ev.get("verdict"),
        })
    
    # Chuyển thành DataFrame và lưu ra file CSV
    eval_df = pd.DataFrame(flat)
    eval_csv = output_dir / "llm_judge_results_flat.csv"
    eval_df.to_csv(eval_csv, index=False)
    print(f"✓ Đã lưu file kết quả CSV tại: {eval_csv}")

    # Tính điểm trung bình
    print("\n📊 TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ (Điểm Clinical Accuracy trung bình):")
    for model in ["svm", "lstm", "bert"]:
        col = f"{model}_clinical_score"
        if col in eval_df.columns:
            # Chuyển đổi sang số thực để tránh lỗi nếu LLM trả về string
            mean_score = pd.to_numeric(eval_df[col], errors='coerce').mean()
            print(f"  - {model.upper():8s}: {mean_score:.2f}/10")
            
else:
    print(f"❌ Không tìm thấy file tại {json_path}. Bạn hãy kiểm tra lại đường dẫn dataset nhé!")


# 📊 Visualize LLM-as-a-Judge Evaluations
Đoạn code dưới đây sẽ đọc file `llm_judge_results_flat.csv` và vẽ các biểu đồ phân tích chuyên sâu.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

output_dir = Path("/kaggle/working")
eval_csv = output_dir / "llm_judge_results_flat.csv"

try:
    eval_df = pd.read_csv(eval_csv)
    
    # 1. Bar Chart: Average Clinical Accuracy
    plt.figure(figsize=(9, 5))
    models = ['SVM', 'Bi-LSTM', 'BERT']
    scores = [
        eval_df['svm_clinical_score'].mean(),
        eval_df['lstm_clinical_score'].mean(),
        eval_df['bert_clinical_score'].mean()
    ]
    
    colors = ['#4C72B0', '#55A868', '#C44E52']
    bars = plt.bar(models, scores, color=colors, alpha=0.9, edgecolor='white')
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f}/10', ha='center', va='bottom', fontweight='bold', fontsize=11)
        
    plt.ylim(0, 11)
    plt.title('Average Clinical Accuracy (Scored by LLM-as-a-Judge)', fontsize=14, fontweight='bold', pad=15)
    plt.ylabel('Clinical Score (0-10)', fontsize=12)
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.savefig(output_dir / 'llm_eval_clinical_accuracy_bar.png', dpi=300, bbox_inches='tight')
    plt.show()

    # 2. Boxplot: Clinical Accuracy by Linguistic Complexity
    melted_df = pd.melt(eval_df, id_vars=['linguistic_complexity'], 
                        value_vars=['svm_clinical_score', 'lstm_clinical_score', 'bert_clinical_score'],
                        var_name='Model', value_name='Clinical_Score')
    melted_df['Model'] = melted_df['Model'].map({'svm_clinical_score':'SVM', 'lstm_clinical_score':'Bi-LSTM', 'bert_clinical_score':'BERT'})
    
    # Clean and sort complexity logically
    melted_df['linguistic_complexity'] = melted_df['linguistic_complexity'].str.capitalize().str.strip()
    complexity_order = ['Low', 'Medium', 'High']
    
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='linguistic_complexity', y='Clinical_Score', hue='Model', data=melted_df, order=complexity_order, palette=colors)
    plt.title('Clinical Accuracy vs Linguistic Complexity', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Linguistic Complexity', fontsize=12)
    plt.ylabel('Clinical Score (0-10)', fontsize=12)
    plt.ylim(0, 11)
    plt.legend(title='Model', fontsize=10)
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.savefig(output_dir / 'llm_eval_complexity_boxplot.png', dpi=300, bbox_inches='tight')
    plt.show()

    # 3. Pie Chart: Model Verdict (Which model won?)
    def get_winner(verdict):
        v = str(verdict).lower()
        if 'bert' in v: return 'BERT'
        if 'lstm' in v or 'bi-lstm' in v: return 'Bi-LSTM'
        if 'svm' in v: return 'SVM'
        return 'Tie/Other'
        
    eval_df['winner'] = eval_df['verdict'].apply(get_winner)
    
    plt.figure(figsize=(7, 7))
    winner_counts = eval_df['winner'].value_counts()
    color_map = {'BERT': '#C44E52', 'Bi-LSTM': '#55A868', 'SVM': '#4C72B0', 'Tie/Other': 'gray'}
    pie_colors = [color_map.get(w, 'gray') for w in winner_counts.index]
    
    plt.pie(winner_counts, labels=winner_counts.index, autopct='%1.1f%%', startangle=90, colors=pie_colors, explode=[0.05]*len(winner_counts), textprops={'fontsize': 12})
    plt.title('LLM Verdict: Most Reliable Model', fontsize=14, fontweight='bold', pad=20)
    plt.savefig(output_dir / 'llm_eval_verdict_pie.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 4. Histogram: Missed Risk Severity
    plt.figure(figsize=(9, 5))
    sns.histplot(eval_df['missed_risk_severity'], bins=10, kde=True, color='#8E44AD')
    plt.title('Distribution of Missed Risk Severity in Edge Cases', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Severity Score (0-10) [10 = Most Dangerous Miss]', fontsize=12)
    plt.ylabel('Number of Samples', fontsize=12)
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.savefig(output_dir / 'llm_eval_risk_severity_hist.png', dpi=300, bbox_inches='tight')
    plt.show()

except FileNotFoundError:
    print(f"\u274c Lỗi: Không tìm thấy file {eval_csv}.")
    print("Vui lòng đảm bảo bạn đã chạy đoạn code parse file JSON sang CSV ở trên!")
except Exception as e:
    print(f"\u274c Lỗi không xác định: {e}")
